In [ ]:
from theia.detection.pcl import PclDetector
from theia.grids import LatLonHeightGrid
from theia.test_data import load_pcl_example_sensors

RCS = 1.0

sensors, lats, lons, alts = load_pcl_example_sensors()
sensor = sensors[1]

grid = LatLonHeightGrid(
    lat_start=lats[0],
    lat_stop=lats[-1],
    lat_res=lats[1] - lats[0],
    lon_start=lons[0],
    lon_stop=lons[-1],
    lon_res=lons[1] - lons[0],
    height_start=alts[0],
    height_stop=alts[-1],
    height_res=alts[1] - alts[0] if len(alts) > 1 else 1.0,
)

In [ ]:
import cProfile

detector = PclDetector()

profiler = cProfile.Profile()

profiler.enable()

min_detectable_rcs = detector.minimum_detectable_rcs_grid(
    sensor.receiver,
    sensor.transmitter,
    grid,
)

profiler.disable()
profiler.dump_stats("profile__min_detectable_rcs.prof")

In [ ]:
from matplotlib import pyplot as plt


plt.imshow(min_detectable_rcs[::-1, :, 0] <= RCS)

In [ ]:
import folium

from theia.mapping import RadarMap
from theia.util import mask_to_polygon

polygons = mask_to_polygon(
    min_detectable_rcs[:, :, 0] <= RCS,
    lats[0],
    lats[1] - lats[0],
    lons[0],
    lons[1] - lons[0],
)

map = RadarMap(sensors={"PCL sensor": sensor}).to_map()
map.location = (sensor.receiver.lat, sensor.receiver.lon)
for polygon in polygons:
    folium.GeoJson(polygon).add_to(map)
map

In [ ]:
import numpy as np

from theia.detection.pcl import PclDetector
from theia.export_paraview import ParaviewExporter, PointOfInterest


pois: list[PointOfInterest] = []
poi_id = 0
for sensor in pcl_sensors:
    pois.append(
        PointOfInterest(
            id=poi_id,
            label=f"Rx {sensor.receiver.id}",
            type="Rx",
            lat=sensor.receiver.lat,
            lon=sensor.receiver.lon,
            alt=sensor.receiver.alt,
        )
    )
    poi_id += 1
    pois.append(
        PointOfInterest(
            id=poi_id,
            label=f"Tx {sensor.transmitter.id}",
            type="Tx",
            lat=sensor.transmitter.lat,
            lon=sensor.transmitter.lon,
            alt=sensor.transmitter.alt,
        )
    )
    poi_id += 1

exporter = ParaviewExporter(
    lat_min,
    lat_max,
    lat_res,
    lon_min,
    lon_max,
    lon_res,
)

sensor = pcl_sensors[0]

detector = PclDetector()

exporter.export_terrain(
    "test",
    # {
    #     "min detectable RCS": lambda *p: detector.minimum_detectable_rcs_vector(
    #         sensor.receiver, sensor.transmitter, np.array(p).reshape((1, 3))
    #     )
    # },
)
exporter.export_pois(pois, "pois.csv")

In [ ]:
import itertools

import pandas as pd

from theia.coordinates import CoordinateTransformations
from theia.terrain import elevationAt


# heatmap2 = np.empty((len(heatmap), 6), dtype=np.float64)
# heatmap2[:, :4] = np.stack(heatmap, axis=-1).T
# for i, row in enumerate(heatmap):
#     p_ecef = np.array(
#         CoordinateTransformations.geodetic_to_cartesian(
#             row[0], row[1], row[2] * exporter._elevation_factor
#         )
#     )
#     p_ecef -= exporter._center
#     p_ecef = (exporter._rotation @ p_ecef.T).T
#     heatmap[i][4:] = p_ecef

# df = pd.DataFrame(heatmap, columns=["lat", "lon", "alt", "min_detectable_rcs", "x", "y", "z"])
# df.to_csv("heatmap.csv")

In [ ]:
from matplotlib import pyplot as plt

for sensor in pcl_sensors:
    minimum_detectable_rcs = np.empty(
        (len(lats), len(lons), len(alts)), dtype=np.float32
    )
    minimum_detectable_rcs[:, :, :] = np.nan
    for i, lat in enumerate(lats):
        for j, lon in enumerate(lons):
            for k, alt in enumerate(alts):
                if alt <= elevationAt(lat, lon):
                    continue
                minimum_detectable_rcs[i, j, k] = (
                    detector.minimum_detectable_rcs_vector(
                        sensor.receiver,
                        sensor.transmitter,
                        np.array([[lat, lon, alt]]),
                    )[0]
                )

    fig, ax = plt.subplots(figsize=(8, 4.5))
    img = ax.imshow(minimum_detectable_rcs[:, :, 0], vmin=0, vmax=3)
    ax.set_xlabel("lon [°]")
    ax.set_ylabel("lat [°]")
    ax.set_xticks(np.arange(len(lons))[::10])
    ax.set_yticks(np.arange(len(lats))[::10])
    ax.set_xticklabels([f"{lon:.3f}" for lon in lons][::10])
    ax.set_yticklabels([f"{lat:.3f}" for lat in lats][::10])
    ax.tick_params(axis="x", rotation=90)
    ax.set_title(f"Minimum detectable RCS for sensor {sensor.id}")
    fig.colorbar(img)
    fig.tight_layout()

In [ ]:
# from theia.detection.pcl import PclDetector
# from theia.terrain import elevationAt
# p = np.array([[47.1497, 47.5351, elevationAt(47.1497, 8.0641)]])

In [ ]:
# %%timeit
# detector.minimum_detectable_rcs_vector(
#     sensor.receiver,
#     sensor.transmitter,
#     p,
# )

In [ ]:
from theia.coverage import calculate_coverage
from theia.radar_equation import calculate_maximum_monostatic_range

calculate_maximum_monostatic_range(radar)

# calculate_coverage(radar.receiver.point, 1)

In [ ]:
radar.receiver.cpi_pulses

In [ ]:
set(d.keys()).union(set(["miaus"]))

In [ ]:
"Marketing" in d

In [ ]:
# from theia.detection.pcl import PclDetector
# from theia.grids import LatLonHeightGrid
# from matplotlib import pyplot as plt

# grid = LatLonHeightGrid(
#     lat_start=47.1497,
#     lat_stop=47.5351,
#     lat_res=0.01,
#     lon_start=8.0641,
#     lon_stop=8.8636,
#     lon_res=0.01,
#     height_start=1000,
#     height_stop=1000,
#     height_res=1,
# )

# for sensor in pcl_sensors[:10]:
#     detector = PclDetector()
#     min_detectable_rcs = detector.minimum_detectable_rcs_vector(
#         sensor.receiver, sensor.transmitter, grid.points
#     ).reshape(grid.n_points)

#     fig, ax = plt.subplots()

#     img = ax.imshow(min_detectable_rcs[:, :, 0], vmin=0, vmax=15)
#     fig.colorbar(img)